# License Plate OCR + Parking Garage Database Pipeline

## Objective

This notebook validates the second stage of the ANPR system:

1. Receive cropped license plate images from the YOLO detector
2. Apply OCR to extract license plate text
3. Store and query vehicle authorization records
4. Determine whether detected vehicles belong to the parking garage

Pipeline:

Vehicle Image Crop
→ OCR
→ License Plate String
→ Database Lookup
→ Access Decision

In [1]:
!pip install paddlepaddle-gpu paddleocr
!pip uninstall -y paddleocr paddlex
!pip install paddleocr==2.8.1

Found existing installation: paddleocr 2.8.1
Uninstalling paddleocr-2.8.1:
  Successfully uninstalled paddleocr-2.8.1
  Using cached paddleocr-2.8.1-py3-none-any.whl.metadata (19 kB)
Using cached paddleocr-2.8.1-py3-none-any.whl (407 kB)


In [2]:

#Mount Google Drive
from google.colab import drive
from pprint import pprint
drive.mount("/content/drive", force_remount=True)

#Sanity Check on mount
import os
print("Contents of my Google Drive")

pprint(os.listdir("/content/drive/MyDrive")[:10])

Mounted at /content/drive
Contents of my Google Drive
['Accident Report.gdoc',
 'God essay.gdoc',
 'get it get it.docx.gdoc',
 'ID LIST for HIST 355.gdoc',
 'Midterm Review, HIST 447.gdoc',
 'projectstat4.gsheet',
 'google docs test.gsheet',
 'j.1745-4549.1997.tb00792.x.pdf',
 'bank shot',
 'jqas.2011.7.1.1299.pdf']


In [3]:
%cd /content/drive/MyDrive/LPR_Project/MD_LicensePlate_Fadiran_ITAI1378/
!git pull

/content/drive/MyDrive/LPR_Project/MD_LicensePlate_Fadiran_ITAI1378
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 12.16 MiB | 5.96 MiB/s, done.
From https://github.com/Toyin-Fadiran/MD_LicensePlate_Fadiran_ITAI1378
   e28c22c..35b7fe1  main       -> origin/main
Updating e28c22c..35b7fe1
Fast-forward
 notebooks/Tier_01_detection.ipynb | 3014 +++++++++++++++++++++++++++++++++++++
 1 file changed, 3014 insertions(+)
 create mode 100644 notebooks/Tier_01_detection.ipynb


In [4]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [5]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/LPR_Project/MD_LicensePlate_Fadiran_ITAI1378"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Added to Python path:")
print(PROJECT_ROOT)

Added to Python path:
/content/drive/MyDrive/LPR_Project/MD_LicensePlate_Fadiran_ITAI1378


In [6]:
%cd /content/drive/MyDrive/LPR_Project/MD_LicensePlate_Fadiran_ITAI1378
#!ls


/content/drive/MyDrive/LPR_Project/MD_LicensePlate_Fadiran_ITAI1378


In [7]:
import glob
from src.ocr.reader import PlateReader
from src.config import (
      RUN_DIR
)

# 1. Initialize your reader
reader = PlateReader()

# 2. Point to the exact folder where YOLO saved your crops
#crop_dir = "/content/drive/MyDrive/LPR_Project/MD_LicensePlate_Fadiran_ITAI1378/runs/detect/predict/crops/License_Plate"
crop_dir = str(RUN_DIR)
# 3. Get a list of all the .jpg files in that folder
crop_files = glob.glob(f"{crop_dir}/*.jpg")

# 4. Loop through and read them!
for image_path in crop_files:
    print(f"\nReading: {image_path.split('/')[-1]}")

    # Pass the file path directly to EasyOCR
    ocr_results = reader.read(image_path)

    for result in ocr_results:
        #print(f"Found Plate: {result['plate']} (Confidence: {result['confidence']:.2f})")
        # To print the bounding box as well:
        print(f"Found Plate: {result['plate']} | Confidence: {result['confidence']:.2f} | BBox: {result['bbox']}")

download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 4.00M/4.00M [00:30<00:00, 132kiB/s]


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10.2M/10.2M [00:20<00:00, 502kiB/s] 


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2.19M/2.19M [00:49<00:00, 44.5kiB/s]



Reading: image0.jpg
Found Plate: PPX8975 | Confidence: 0.95 | BBox: [[37, 77], [261, 84], [259, 142], [35, 135]]

Reading: image0-2.jpg
Found Plate: GNX3211 | Confidence: 0.94 | BBox: [[36, 63], [242, 63], [242, 113], [36, 113]]

Reading: image1.jpg
Found Plate: WYC5321 | Confidence: 0.97 | BBox: [[47, 89], [301, 89], [301, 154], [47, 154]]

Reading: image1-2.jpg
Found Plate: JTS6564 | Confidence: 0.98 | BBox: [[43, 85], [288, 80], [289, 141], [45, 146]]

Reading: image2.jpg
Found Plate: KFM5189 | Confidence: 0.92 | BBox: [[114, 191], [644, 185], [645, 315], [115, 321]]

Reading: image3.jpg
Found Plate: MRG7962 | Confidence: 0.96 | BBox: [[63, 131], [513, 126], [514, 230], [64, 235]]

Reading: image4.jpg
Found Plate: SXJ8586 | Confidence: 0.97 | BBox: [[113, 168], [587, 183], [584, 308], [109, 293]]

Reading: image5.jpg
Found Plate: FCF6902 | Confidence: 0.95 | BBox: [[95, 167], [623, 167], [623, 301], [95, 301]]

Reading: image6.jpg
Found Plate: DSV3638 | Confidence: 0.94 | BBox: [[9

In [8]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# Import your custom modules
from src.config import RUN_DIR
from src.ocr.reader import PlateReader
from src.database.db import Database
from src.database.queries import seed_database, check_vehicle, log_detection

print("1. Initializing System & Database Infrastructure...")
# Because we used default arguments, this automatically connects
# to db_data/lpr.db and uses your authorized_list.csv!
db = Database()
db.create_tables()

# 2. Seed the authorized list
seed_database(db)

# 3. Initialize the OCR Engine
print("\n2. Waking up the OCR Engine...")
reader = PlateReader()

# 4. Locate all cropped plates from YOLO
# Since RUN_DIR is a Path object, we can use .glob() directly
crop_files = list(RUN_DIR.glob("*.jpg"))
print(f"Found {len(crop_files)} license plates to process.")

print("\n3. Starting Real-Time Processing Stream...")
print("-" * 50)

# 5. The Core Loop
for image_path in crop_files:
    # Convert Path object to string for the cv2/EasyOCR reader
    image_str = str(image_path)
    file_name = image_path.name

    # Run the dynamic 2-pass OCR
    ocr_results = reader.read(image_str)

    if not ocr_results:
        print(f"[{file_name}] - No readable text found.")
        continue

    for result in ocr_results:
        plate_text = result['plate']
        conf = result['confidence']

        # A. Database Check
        auth_record = check_vehicle(db, plate_text)
        is_authorized = bool(auth_record)

        # B. Log to DB
        log_detection(db, plate_text, conf, is_authorized)

        # C. Console Output
        status = "✅ AUTHORIZED" if is_authorized else "🚨 UNKNOWN"
        print(f"[{file_name}] | Plate: {plate_text} | Conf: {conf:.2f} | Status: {status}")

print("-" * 50)
print("\n4. Pipeline Complete. Generating Security Snapshot...\n")

# 6. Generate the Pandas Snapshot of Unauthorized Vehicles
query = """
    SELECT license_plate, ocr_confidence, detected_at
    FROM detections
    WHERE authorized = 0
    ORDER BY detected_at DESC
"""
unauthorized_df = pd.read_sql_query(query, db.connection)

if unauthorized_df.empty:
    print("All detected vehicles were authorized!")
else:
    print("🚨 UNAUTHORIZED VEHICLES LOG 🚨")
    display(unauthorized_df)

# 7. Safely close connection
db.close()

1. Initializing System & Database Infrastructure...
Loading authorized vehicles from /content/drive/MyDrive/LPR_Project/MD_LicensePlate_Fadiran_ITAI1378/authorized_list.csv...
Successfully seeded 20 vehicles!

2. Waking up the OCR Engine...
Found 50 license plates to process.

3. Starting Real-Time Processing Stream...
--------------------------------------------------
[image0.jpg] | Plate: PPX8975 | Conf: 0.95 | Status: 🚨 UNKNOWN
[image0-2.jpg] | Plate: GNX3211 | Conf: 0.94 | Status: ✅ AUTHORIZED
[image1.jpg] | Plate: WYC5321 | Conf: 0.97 | Status: ✅ AUTHORIZED
[image1-2.jpg] | Plate: JTS6564 | Conf: 0.98 | Status: 🚨 UNKNOWN
[image2.jpg] | Plate: KFM5189 | Conf: 0.92 | Status: ✅ AUTHORIZED
[image3.jpg] | Plate: MRG7962 | Conf: 0.96 | Status: 🚨 UNKNOWN
[image4.jpg] | Plate: SXJ8586 | Conf: 0.97 | Status: 🚨 UNKNOWN
[image5.jpg] | Plate: FCF6902 | Conf: 0.95 | Status: 🚨 UNKNOWN
[image6.jpg] | Plate: DSV3638 | Conf: 0.94 | Status: ✅ AUTHORIZED
[image7.jpg] | Plate: PJP4382 | Conf: 0.93 | 

,license_plate,ocr_confidence,detected_at
0,MXW4752,0.919461,2026-07-29 03:09:49.295523
1,KPX7625,0.969360,2026-07-29 03:09:49.149175
2,PMV0645,0.947903,2026-07-29 03:09:49.066554
3,KWM0999,0.939682,2026-07-29 03:09:48.993291
4,PPX7282,0.972278,2026-07-29 03:09:48.906653
5,VYB0416,0.935002,2026-07-29 03:09:48.841524
6,GBL8203,0.951265,2026-07-29 03:09:48.585933
7,BABY,0.995240,2026-07-29 03:09:48.382575
8,BABY,0.995796,2026-07-29 03:09:48.346198
9,PLANETK,0.742714,2026-07-29 03:09:48.223268
